# Set Transformer para roles posicionales

Este notebook permite dos modos:
- usar un checkpoint ya entrenado y hacer solo inferencia
- reentrenar el modelo y después aplicar el checkpoint nuevo

Opcionalmente puedes fijar el once esperado de cada equipo con `EXPECTED_ROLES_BY_TEAM`: la predicción estable ya no se decide jugador a jugador de forma independiente, sino con una asignación global tipo Hungarian que impone las plazas permitidas por equipo.

Además, tras predecir, el notebook puede renderizar automáticamente el vídeo anotado con `role`.

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config.yaml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if not (PROJECT_ROOT / "config.yaml").exists():
    raise FileNotFoundError("No se encontró config.yaml al subir desde el cwd del notebook.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import experiments.positions.set_transformer_pipeline as st_pipeline
st_pipeline = importlib.reload(st_pipeline)

from experiments.positions.set_transformer_pipeline import (
    TrainingConfig,
    predict_roles_for_video,
    render_role_video,
    train_position_model,
)

BASE_TABLE_PATH = PROJECT_ROOT / "data" / "posiciones_etiquetadas" / "common" / "base_table.csv"
#SOURCE_VIDEO_PATH = PROJECT_ROOT / "data" / "partidoPrueba" / "partido.mp4"
VIDEO_PATH = PROJECT_ROOT / "data" / "partidoPrueba" / "partido_ajustado.mp4"
#CLIP_START_FRAME = 60

USE_EXISTING_CHECKPOINT = True
MODEL_PATH = PROJECT_ROOT / "models" / "positions" / "set_transformer" / "20260317_211507" / "set_transformer_checkpoint.pt"
RENDER_ANNOTATED_VIDEO = True

EXPECTED_ROLES_BY_TEAM = {
    # Usa los team_id detectados en tracks/predicciones.
    # Admite labels del modelo y algunos aliases prácticos:
    # DEL -> DC, DFIZQ -> DFC_IZQ, DFDCHA -> DFC_DER, DFCENT -> DFC_CENT.
    "Real Madrid": ["POR", "CD", "CI", "DFC_DER", "DFC_IZQ", "DFC_CENT", "MC", "MI", "MD", "DC", "DC"],
    "Wolfsburgo": ["POR", "CD", "CI", "DFC_DER", "DFC_IZQ", "DFC_CENT", "MC", "MI", "MD", "DC", "DC"],
}

TRAIN_CONFIG = TrainingConfig(
    epochs=18,
    batch_size=512,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=5,
    seed=42,
)

PROJECT_ROOT, BASE_TABLE_PATH.exists(),  VIDEO_PATH.exists(),  USE_EXISTING_CHECKPOINT, MODEL_PATH, RENDER_ANNOTATED_VIDEO


(PosixPath('/Users/carloscole/narrador-futbol'),
 True,
 True,
 True,
 PosixPath('/Users/carloscole/narrador-futbol/models/positions/set_transformer/20260317_211507/set_transformer_checkpoint.pt'),
 True)

In [2]:
if USE_EXISTING_CHECKPOINT:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"No existe el checkpoint configurado: {MODEL_PATH}")
    training_result = None
    checkpoint_path = MODEL_PATH
else:
    training_result = train_position_model(
        project_root=PROJECT_ROOT,
        base_table_path=BASE_TABLE_PATH,
        config=TRAIN_CONFIG,
    )
    checkpoint_path = training_result["checkpoint_path"]

checkpoint_path


PosixPath('/Users/carloscole/narrador-futbol/models/positions/set_transformer/20260317_211507/set_transformer_checkpoint.pt')

In [3]:
if training_result is None:
    display(
        pd.DataFrame(
            [
                {
                    "mode": "existing_checkpoint",
                    "checkpoint_path": str(checkpoint_path),
                }
            ]
        )
    )
else:
    history_df = training_result["history_df"]
    display(history_df.tail())


,mode,checkpoint_path
0,existing_checkpoint,/Users/carloscole/narrador-futbol/models/posit...


In [4]:
prediction_result = predict_roles_for_video(
    model_path=checkpoint_path,
    video_path=VIDEO_PATH,
    project_root=PROJECT_ROOT,
    expected_roles_by_team=EXPECTED_ROLES_BY_TEAM or None,
)

prediction_result["output_dir"]


PosixPath('/Users/carloscole/narrador-futbol/output/predictions/positions/partido_ajustado_20260318_121702')

In [5]:
if RENDER_ANNOTATED_VIDEO:
    render_result = render_role_video(
        video_path=VIDEO_PATH,
        tracks_path=prediction_result["tracks_with_roles_path"],
        project_root=PROJECT_ROOT,
    )
    render_result["video_path"]
else:
    render_result = None
    print("Render desactivado.")


In [6]:
player_summary = pd.read_csv(prediction_result["player_predictions_path"])
frame_predictions = pd.read_csv(prediction_result["frame_predictions_path"])

if render_result is not None:
    display(pd.DataFrame([{"annotated_video_path": str(render_result["video_path"])}]))

display(
    player_summary.sort_values(["team_id", "player_id"])[
        [
            "team_id",
            "player_id",
            "class_name",
            "predicted_role_unconstrained",
            "predicted_role",
            "expected_role_slot",
            "assignment_method",
            "predicted_role_confidence",
        ]
    ].head(30)
)
display(
    frame_predictions[
        [
            "frame_id",
            "team_id",
            "player_id",
            "class_name",
            "predicted_role_frame",
            "predicted_role_unconstrained",
            "predicted_role",
            "expected_role_slot",
        ]
    ].head(30)
)


,annotated_video_path
0,/Users/carloscole/narrador-futbol/output/predi...


,team_id,player_id,class_name,predicted_role_unconstrained,predicted_role,expected_role_slot,assignment_method,predicted_role_confidence
0,Real Madrid,2,player,LD,DFC_DER,DFC_DER,hungarian_expected_roles,0.102722
1,Real Madrid,4,player,DFC_DER,DFC_CENT,DFC_CENT,hungarian_expected_roles,0.004461
2,Real Madrid,5,player,EI,MI,MI,expected_roles_fallback_best_allowed,0.285731
3,Real Madrid,10,player,ED,CD,CD,hungarian_expected_roles,0.000349
4,Real Madrid,11,player,DC,DC,DC,hungarian_expected_roles,0.447280
5,Real Madrid,16,player,MC,MC,MC,hungarian_expected_roles,0.694966
6,Real Madrid,17,player,DFC_IZQ,DFC_IZQ,DFC_IZQ,hungarian_expected_roles,0.578022
7,Real Madrid,19,player,MI,MI,MI,expected_roles_fallback_best_allowed,0.466979
8,Real Madrid,20,player,MI,MI,MI,hungarian_expected_roles,0.796451
9,Real Madrid,21,player,DC,DC,DC,hungarian_expected_roles,0.900053


,frame_id,team_id,player_id,class_name,predicted_role_frame,predicted_role_unconstrained,predicted_role,expected_role_slot
0,5,Real Madrid,2,player,LD,LD,DFC_DER,DFC_DER
1,5,Real Madrid,4,player,DFC_DER,DFC_DER,DFC_CENT,DFC_CENT
2,5,Real Madrid,5,player,EI,EI,MI,MI
3,5,Real Madrid,10,player,ED,ED,CD,CD
4,5,Real Madrid,11,player,MI,DC,DC,DC
5,5,Real Madrid,16,player,MC,MC,MC,MC
6,5,Real Madrid,17,player,LI,DFC_IZQ,DFC_IZQ,DFC_IZQ
7,5,Real Madrid,19,player,MI,MI,MI,MI
8,5,Real Madrid,20,player,MI,MI,MI,MI
9,5,Real Madrid,21,player,DC,DC,DC,DC
